## PDF Chunking Evaluation for Chatbot
### Use case: Annual Report Chabot

You are building a chatbot that can answer questions from an annual report PDF. To ensure efficeint and accurate retrieval by the LLM, you need to chunk the document into semantically meanigful and size optimized sections.

The notebook walks thorugh multiple chunking strategies and helps identify the best method on semantic coherence and retreival performance.

The provided pdf is Annual Report for Mondelez in the year 2001.

#### Objectives:
- Load a sample PDF (annual report)
- Apply mutliple chunking strategies:
    - Fixed-size chunking
    - Sentence-based chunking
    - Semantic/Context based chunking
- Evaluate and compare chunking methods

##### Import necessary libraries

In [2]:
!pip install PyMuPDF
!pip install nltk scikit-learn sentence-transformers faiss-cpu numpy langchain  langchain-experimental

In [3]:
import fitz
import nltk
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
import faiss

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [4]:
import os
from google.colab import userdata

os.environ['HUGGINGFACE_HUB_TOKEN'] = userdata.get('HF_TOKEN')

In [5]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

##### Read the unstructured file (PDFs)

In [6]:
def load_pdf_text(pdf_path: str) -> str:
    '''
    Extract text from a PDF file using PyMuPDF (fitz).

    Args:
        pdf_path: Path to the PDF file

    Returns:
        str: The extracted text from the PDF
    '''
    import fitz  # PyMuPDF

    text = ""

    try:
        # Open the PDF file
        doc = fitz.open(pdf_path)

        # Iterate through each page and extract text
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            page_text = page.get_text()
            text += page_text

        # Close the document
        doc.close()

    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return ""

    return text



# print(pdf_text)

In [7]:
from google.colab import drive
drive.mount('/content/drive')

pdf_text = load_pdf_text("/content/drive/MyDrive/annual_report.pdf")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Chunking Strategies

There are three chunking methodolgies provided below
- Fixed-chunking strategy
- Sentence-chunking strategy
- Semantic/Context based chunking strategy


You are required to complete the functions below. You can change the parameters in the functions if required.
    

In [8]:
# Strategy 1: Fixed-size chunking
from typing import List
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter

def fixed_chunking(text: str, max_tokens: int = 500) -> List[str]:
    """
    Split text into fixed-size chunks using LangChain's text splitter.

    Args:
        text: The text to split
        max_tokens: Maximum characters per chunk

    Returns:
        List of text chunks
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_tokens,
        chunk_overlap=100,
        separators=["\n\n", "\n", " ", ""]
    )

    return splitter.split_text(text)


fixed_chunks =  fixed_chunking(pdf_text)

In [9]:
len(fixed_chunks[2])

481

In [10]:
import re
def _split_sentences(text):
    # Use regular expressions to split the text into sentences based on punctuation followed by whitespace.
    sentences = re.split(r'(?<=[.?!])\s+', text)
    return sentences

In [43]:
len(_split_sentences(pdf_text))

1190

In [11]:
def sentence_chunking(sentences, sentences_per_chunk=5):
    """
    Group a list of sentences into chunks containing a specified number of sentences each.

    Args:
        sentences: List of individual sentences
        sentences_per_chunk: Number of sentences to include in each chunk (default: 5)

    Returns:
        List of text chunks, where each chunk contains the specified number of sentences
    """
    chunks = []

    # Process complete chunks
    for i in range(0, len(sentences), sentences_per_chunk):
        # Get a slice of sentences_per_chunk sentences
        sentence_group = sentences[i:i+sentences_per_chunk]

        # Join the sentences into a single text chunk
        chunk = " ".join(sentence_group)

        # Add the chunk to our results
        chunks.append(chunk)

    return chunks

sentence_chunks=sentence_chunking(_split_sentences(pdf_text))

In [12]:
def get_embeddings(sentences):
  return model.encode(sentences)

def _calculate_cosine_distances(embeddings):
    # Calculate the cosine distance (1 - cosine similarity) between consecutive embeddings.
    distances = []
    for i in range(len(embeddings) - 1):
        similarity = cosine_similarity([embeddings[i]], [embeddings[i + 1]])[0][0]
        distance = 1 - similarity
        distances.append(distance)
    return distances

In [49]:
def semantic_chunking(text):
  sentences=_split_sentences(pdf_text)
  embeddings=get_embeddings(sentences)
  distances=_calculate_cosine_distances(embeddings)
  breakpoint_percentile_threshold =80
  breakpoint_distance_threshold = np.percentile(distances, breakpoint_percentile_threshold)
  indices_above_thresh = [i for i, distance in enumerate(distances) if distance > breakpoint_distance_threshold]

  chunks = []
  start_index = 0
  # Loop through the identified breakpoints and create chunks accordingly.
  for index in indices_above_thresh:
      chunk = ' '.join(sentences[start_index:index+1])
      chunks.append(chunk)
      start_index = index + 1

  # If there are any sentences left after the last breakpoint, add them as the final chunk.
  if start_index < len(sentences):
      chunk = ' '.join(sentences[start_index:])
      chunks.append(chunk)

  return chunks

In [50]:
semantic_chunks=semantic_chunking(pdf_text)

In [72]:
semantic_chunks[18]

"Growth is the\nheart of our culture. We’re committed to driving shareholder value \nwith consistent, top-tier performance. 14\n*For a more meaningful comparison, results are presented on a pro forma basis, \nand volume results are compared to a 52-week fiscal year 2000. See 2001 Financial \nHighlights on page18 for a more detailed explanation. Volume\nOperating Companies Income\nNet Earnings\nTotal Shareholder Return\n3.7%\n2001\n2000\n19.9%\n2001\n2000\n10.6%\n12/31/01\n6/13/01\n8.9%\n2001\n2000\n52-week\nPro Forma*\nPro Forma*\nPro Forma*\nA.1., AIR CRISPS, ALMOST HOME, ALPHA-BITS, ALTOIDS, ATHENOS, BAKER'S, BALANCE NUTRITION BAR, BANANA NUT CRUNCH, BARNUM'S ANIMALS, BETTER C\nLUEBERRY MORNING, BOCA, BREAKSTONE'S, BULL'S-EYE, CAFÉ CREMES, CALUMET, CAMEO, CAPRI SUN, CARRIAGE INN, CARTE NOIRE, CASINO, CASTELET\nHEESE NIPS CRISPS, CHEESE PUFFS, CHEEZ WHIZ, CHIPS AHOY!, CHURNY, CINNA-CRUNCH PEBBLES, CLIGHT, CLUB SOCIAL, COOL WHIP, CORNNUTS, CÔTE D’OR\nME, CRACKER BARREL, CRANBERRY ALMON

In [89]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

def create_faiss_index(chunks, model, query, top_k=3):
  top_k=3
  chunk_embeddings = model.encode(chunks, convert_to_numpy=True)

      # Step 2: Build the FAISS index
  dimension = chunk_embeddings.shape[1]
  index = faiss.IndexFlatL2(dimension)
  index.add(chunk_embeddings)

  # Step 3: Encode the query
  query_embedding = model.encode([query], convert_to_numpy=True)

  # Step 4: Search the index
  D, I = index.search(query_embedding, top_k)

  # Step 5: Print the results
  print(f"Top {top_k} matching chunks for: '{query}'\n")
  for i in I[0]:
      print(chunks[i])
      print("-" * 80)

In [90]:
create_faiss_index(fixed_chunks, model,"what is the reported volume for oscar mayer and pizza in 2001?")

Top 3 matching chunks for: 'what is the reported volume for oscar mayer and pizza in 2001?'

increased 4.7%.
Oscar Mayer and Pizza: Reported volume in 2000 increased
5.2% from 1999. Volume grew in pizza, reflecting the continued
success of rising crust pizza and new product introductions.
Volume growth also reflected the introduction of new lunch
combination varieties, the acquisition of Boca Burger, Inc. and
gains in hot dogs and cold cuts. On an underlying basis, volume
increased 5.9%, of which approximately 0.8 percentage points
related to the acquisition of Boca Burger, Inc.
--------------------------------------------------------------------------------
On a pro forma basis, operating companies income increased 7.0%.
Oscar Mayer and Pizza: Reported volume in 2001 increased
0.8% over 2000. Excluding the 53rd week of shipments in 2000,
volume increased 2.3%, due to volume gains in processed meats
and pizza. The processed meats business recorded volume gains
in luncheon meats, hot do

In [91]:
create_faiss_index(sentence_chunks, model,"what is the reported volume for oscar mayer and pizza in 2001?")

Top 3 matching chunks for: 'what is the reported volume for oscar mayer and pizza in 2001?'

On a pro forma basis, operating companies income increased 7.0%. Oscar Mayer and Pizza: Reported volume in 2001 increased
0.8% over 2000. Excluding the 53rd week of shipments in 2000,
volume increased 2.3%, due to volume gains in processed meats
and pizza. The processed meats business recorded volume gains
in luncheon meats, hot dogs, bacon and soy-based meat
alternatives. Volume in the pizza business increased, driven by
new products.
--------------------------------------------------------------------------------
($113 million), partially offset by the
shift in CDC revenues ($44 million). Reported operating companies income increased $81 million (8.0%)
over 1999, due primarily to higher volume/mix ($50 million), the
1999 separation charges ($46 million), higher margins ($20 million,
due primarily to lower commodity costs) and the acquisition of
Balance Bar Co., partially offset by higher mark

In [92]:
create_faiss_index(semantic_chunks, model,"what is the reported volume for oscar mayer and pizza in 2001?")

Top 3 matching chunks for: 'what is the reported volume for oscar mayer and pizza in 2001?'

In addition, the volume comparisons
are also adjusted to reflect a 52-week fiscal year 2000. 20
Cheese, Meals and Enhancers – Volume grew 0.9%, as growth in Meals and
Enhancers and in Canada more than offset declines in Cheese and Food Service,
due in part to exiting non-branded businesses. Operating companies income
increased 5.1%. Oscar Mayer and Pizza – Volume was up 2.3% on gains from our processed meats,
meat alternatives and pizza businesses. Operating companies income was up 5.4%.
--------------------------------------------------------------------------------
As a result, the
Company recorded a pre-tax charge of $157 million during 1999. This charge was included in marketing, administration and research
costs in the consolidated statement of earnings for the following
segments: Cheese, Meals and Enhancers, $71 million; Oscar
Mayer and Pizza, $38 million; Biscuits, Snacks and Confectione

## The sentence chunking is the best chunking strategy here.



*   Semantic chunking failed. None of the retrived chunks are relavant
*   Fixed length chunking performed better than Semantic chunking as it retrieved relevant chunks but the most relevant chunk was ranked lower
*   Sentence chunking is the best choice here as it ranked the most relevant chunk at the top

